### Exercício de datas

A sua loja virtual fez uma venda na quinta-feira, dia 09/07/2026 às 15:30:00. O prazo de garantia e devolução é de exatamente 18 dias e 6 horas após a compra.
Em qual data e horário exato expira o prazo dessa garantia?

In [ ]:
from datetime import datetime, timedelta

data_venda = datetime(2026, 7, 9, 15, 30)

tempo_garantia = timedelta(days=18, hours=6)

data_expiracao = data_venda + tempo_garantia

print(data_expiracao)

2026-07-27 21:30:00


Você tem dois registros de horário do mesmo dia:
Entrada na sala: 09:15:30
Saída da sala: 11:42:15
Usando a biblioteca datetime, crie os objetos com essas horas (pode usar o mesmo dia para ambos) e subtraia-os para obter a duração.
Pergunta: Quantos segundos totais durou essa reunião? (Dica: o objeto timedelta resultante tem o método .total_seconds()).


In [ ]:
from datetime import datetime

entrada_sala = datetime(2026,7, 24, 9, 15, 30)
saida_sala = datetime(2026, 7, 24, 11, 42, 15)

duracao = saida_sala - entrada_sala

duracao_segundos = duracao.total_seconds()

print(duracao_segundos)

8805.0


Qual cliente teve o maior número de dias entre o cadastro e a última compra, e de quantos dias foi esse intervalo?


In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "Cliente": ["Ana", "Bruno", "Carla", "Diego"],
        "Data_Cadastro": [
            "2026-01-15",
            "2026-03-22",
            "2026-07-10",
            "2026-07-18",
        ],
        "Data_Ultima_Compra": [
            "2026-07-20",
            "2026-04-01",
            "2026-07-11",
            "2026-07-22",
        ],
    }
)

# 1. Converte as colunas para datetime
df["Data_Cadastro"] = pd.to_datetime(df["Data_Cadastro"])
df["Data_Ultima_Compra"] = pd.to_datetime(df["Data_Ultima_Compra"])

# 2. Crie a coluna 'Dias_Inativo' subtraindo a Cadastro da Ultima_Compra
df['Dias_Inativo'] = df["Data_Ultima_Compra"] -  df["Data_Cadastro"]

df['Dias_Inativo']
# 3. Descubra o resultado!

df



,Cliente,Data_Cadastro,Data_Ultima_Compra,Dias_Inativo
0,Ana,2026-01-15,2026-07-20,186 days
1,Bruno,2026-03-22,2026-04-01,10 days
2,Carla,2026-07-10,2026-07-11,1 days
3,Diego,2026-07-18,2026-07-22,4 days



### Continuando regex com re


In [ ]:
# .
# [] - ou
# | - ou
# [A-Za-z]
# sub - (padrao, substituicao, onde fazer substituicao)
# ()
# re.I, re.IGNORECASE
# ? - opcional
# + 1 ou mais ocorrencias
# * 0 ou mais ocorrencias
# \d - digitos
# \b - borda
# \w -

import re

texto = """Mateus e Matheus comprou 3 carros caros caos em 15/08/1984.
Entre em contato com carlos.silva@email.com ou
pelo telefone (11) 98765-4321 para confirmar os detalhes."
"""

inicio = r'[^A]' # NEGACAO
incio = r'^A' # INICIO DA STRING COM

padrao = r'(\d{2})/(\d{2})/(\d{4})' # ( )


matches = re.findall(padrao, texto)

# re.search() # busca primeira ocorrencia - None

for ocorrencia in re.finditer(padrao, texto):
  print(ocorrencia.group(0))

  dia = ocorrencia.group(1)
  mes = ocorrencia.group(2)
  ano = ocorrencia.group(3)

  print(f"Dia: {dia}, mes: {mes}, ano: {ano}")


15/08/1984
Dia: 15, mes: 08, ano: 1984


In [ ]:
texto2 = "<p>Paragrafo 1</p><p>Paragrafo 2</p><div>Div 1</div><p>A</p>"

padrao2 = r'<p>(.+?)</p>'

print(re.findall(padrao2, texto2))
print(re.search(padrao2, texto2).group(1))

padrao3 = r"\d{2}"
padrao3.findall(texto2)

['Paragrafo 1', 'Paragrafo 2', 'A']
Paragrafo 1


### Regex com pandas

In [ ]:
pip install pandas

In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 1: IMPORTAÇÃO DAS BIBLIOTECAS
# ------------------------------------------------------------------------------
# Importar a biblioteca pandas para manipulação dos dados estruturados
import pandas as pd

In [ ]:
import pandas
# ------------------------------------------------------------------------------
# ETAPA 2: CARREGAMENTO DO ARQUIVO EXCEL
# ------------------------------------------------------------------------------
# Ler o arquivo de vendas usando o pd.read_excel()

df = pandas.read_excel('vendas_brutas_clientes.xlsx', usecols=['ID_Venda', 'Data_Venda'], )

df.head()

,ID_Venda,Data_Venda
0,VND-1001,15/01/2026
1,VND-1002,2026-01-18
2,VND-1003,02/02/2026
3,VND-1004,2026/02/14
4,VND-1005,05-03-2026


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 3: TRATAMENTO DE DATAS COM pd.to_datetime()
# ------------------------------------------------------------------------------
# Converter a coluna 'Data_Venda' (que possui múltiplos formatos) para o tipo datetime nativo
# Usamos dayfirst=True para indicar o padrão brasileiro (dia/mês/ano)
df['Data_Venda_Atualizado'] = pd.to_datetime(df['Data_Venda'],
                                             format='mixed', dayfirst=True)

df.head()
df.dtypes


,0
ID_Venda,object
Data_Venda,object
Valor_Venda,float64
Nome_Vendedor,object
Nome_Cliente,object
Email_Cliente,object
Telefone_Cliente,object
Data_Venda_Atualizado,datetime64[ns]


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 4: LIMPEZA DE TELEFONES COM REGEX (str.replace)
# ------------------------------------------------------------------------------
# Usamos os parênteses em volta da expressão para permitir o encadeamento de métodos (Method Chaining) em múltiplas linhas
# 1. Convertemos para string (.astype(str))
# 2. Removemos o código internacional +55 do início (r'^\+55\s?')
# 3. Removemos caracteres de formatação como parênteses, espaços, pontos e traços (r'[()\s.-]')

df['Telefone_Atualizado'] = (df['Telefone_Cliente'].astype(str)
.str.replace(r'^\+55\s?', '', regex=True)).str.replace(r'[()\s.-]', '', regex=True)


df['Telefone_Atualizado']

,Telefone_Atualizado
0,11987654321
1,21998871122
2,31988776655
3,41912345678
4,11976543210
5,81981112233
6,85994433221
7,61991238899
8,19987123456
9,51984567890


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 5: EXTRAÇÃO DE DDD E NÚMERO COM REGEX (str.extract)
# ------------------------------------------------------------------------------
# Utilizamos grupos de captura com parênteses (...) na Regex:
# - Grupo 1: (\d{2}) -> Captura os 2 primeiros dígitos como DDD
# - Grupo 2: (\d{8,9}) -> Captura os 8 ou 9 dígitos restantes do número

df[['DDD', 'Telefone']] = df['Telefone_Atualizado'].str.extract(r'^(\d{2})(\d{8,9})$')

df.head()

,ID_Venda,Data_Venda,Valor_Venda,Nome_Vendedor,Nome_Cliente,Email_Cliente,Telefone_Cliente,Data_Venda_Atualizado,Telefone_Atualizado,DDD,Telefone
0,VND-1001,15/01/2026,1250.5,Carlos Eduardo,Mariana Souza,mariana.souza@gmail.com,(11) 98765-4321,2026-01-15,11987654321,11,987654321
1,VND-1002,2026-01-18,3400.0,Ana Beatriz,Roberto Alves,roberto_alves@empresa.com.br,+55 21 99887-1122,2026-01-18,21998871122,21,998871122
2,VND-1003,02/02/2026,890.9,Carlos Eduardo,Fernanda Lima,fernanda.lima@invalid_email.c,31988776655,2026-02-02,31988776655,31,988776655
3,VND-1004,2026/02/14,5200.0,Lucas Mendes,João Pedro Santos,joao.pedro@tech.io,(41) 91234.5678,2026-02-14,41912345678,41,912345678
4,VND-1005,05-03-2026,1750.0,Ana Beatriz,Camila Rodrigues,camila.rodrigues@hotmail.com,(11) 97654-3210,2026-03-05,11976543210,11,976543210


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 6: VALIDAÇÃO DE E-MAILS COM REGEX (str.contains)
# ------------------------------------------------------------------------------
# Criamos um padrão para e-mail válido que exige no mínimo 2 letras no final após o ponto final
# Evita aceitar extensões inválidas terminadas com apenas 1 caractere (ex: .c ou .x)
padrao_email_valido = r'[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}$'

df['Email_Valido'] = df['Email_Cliente'].str.contains(padrao_email_valido, regex=True)

df['Email_Valido']
df.head()
# Aplicamos .str.contains() para retornar True (válido) ou False (inválido)



,ID_Venda,Data_Venda,Valor_Venda,Nome_Vendedor,Nome_Cliente,Email_Cliente,Telefone_Cliente,Data_Venda_Atualizado,Telefone_Atualizado,DDD,Telefone,Email_Valido
0,VND-1001,15/01/2026,1250.5,Carlos Eduardo,Mariana Souza,mariana.souza@gmail.com,(11) 98765-4321,2026-01-15,11987654321,11,987654321,True
1,VND-1002,2026-01-18,3400.0,Ana Beatriz,Roberto Alves,roberto_alves@empresa.com.br,+55 21 99887-1122,2026-01-18,21998871122,21,998871122,True
2,VND-1003,02/02/2026,890.9,Carlos Eduardo,Fernanda Lima,fernanda.lima@invalid_email.c,31988776655,2026-02-02,31988776655,31,988776655,False
3,VND-1004,2026/02/14,5200.0,Lucas Mendes,João Pedro Santos,joao.pedro@tech.io,(41) 91234.5678,2026-02-14,41912345678,41,912345678,True
4,VND-1005,05-03-2026,1750.0,Ana Beatriz,Camila Rodrigues,camila.rodrigues@hotmail.com,(11) 97654-3210,2026-03-05,11976543210,11,976543210,True


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 7: FILTRAGEM DE DADOS COMBINADA
# ------------------------------------------------------------------------------
# Filtrar apenas as vendas que possuem e-mail válido

vendas_validas = df[df['Email_Valido'] == True]
vendas_validas

,ID_Venda,Data_Venda,Valor_Venda,Nome_Vendedor,Nome_Cliente,Email_Cliente,Telefone_Cliente,Data_Venda_Atualizado,Telefone_Atualizado,DDD,Telefone,Email_Valido
0,VND-1001,15/01/2026,1250.5,Carlos Eduardo,Mariana Souza,mariana.souza@gmail.com,(11) 98765-4321,2026-01-15,11987654321,11,987654321,True
1,VND-1002,2026-01-18,3400.0,Ana Beatriz,Roberto Alves,roberto_alves@empresa.com.br,+55 21 99887-1122,2026-01-18,21998871122,21,998871122,True
3,VND-1004,2026/02/14,5200.0,Lucas Mendes,João Pedro Santos,joao.pedro@tech.io,(41) 91234.5678,2026-02-14,41912345678,41,912345678,True
4,VND-1005,05-03-2026,1750.0,Ana Beatriz,Camila Rodrigues,camila.rodrigues@hotmail.com,(11) 97654-3210,2026-03-05,11976543210,11,976543210,True
5,VND-1006,12/03/2026,2100.3,Carlos Eduardo,Felipe Oliveira,felipe.oliveira@dominio.org,+55 (81) 98111-2233,2026-03-12,81981112233,81,981112233,True
6,VND-1007,2026-03-22,4300.0,Lucas Mendes,Beatriz Castro,beatriz.castro@br-corporation.com,85994433221,2026-03-22,85994433221,85,994433221,True
7,VND-1008,01/04/2026,650.0,Ana Beatriz,Gabriel Martins,gabriel.m@provedor.com.br,(61) 99123-8899,2026-04-01,61991238899,61,991238899,True
9,VND-1010,28/04/2026,2950.8,Lucas Mendes,Thiago Barbosa,thiago.barbosa@consultoria.com,(51) 98456-7890,2026-04-28,51984567890,51,984567890,True


In [ ]:
# ------------------------------------------------------------------------------
# ETAPA 8: EXIBIÇÃO E CONSOLIDAÇÃO DOS RESULTADOS
# ------------------------------------------------------------------------------
# Seleção das colunas limpas para exibição final formatada
df.tail()
df.describe()


,ID_Venda,Data_Venda,Valor_Venda,Nome_Vendedor,Nome_Cliente,Email_Cliente,Telefone_Cliente,Data_Venda_Atualizado,Telefone_Atualizado,DDD,Telefone,Email_Valido
0,VND-1001,15/01/2026,1250.5,Carlos Eduardo,Mariana Souza,mariana.souza@gmail.com,(11) 98765-4321,2026-01-15,11987654321,11,987654321,True
1,VND-1002,2026-01-18,3400.0,Ana Beatriz,Roberto Alves,roberto_alves@empresa.com.br,+55 21 99887-1122,2026-01-18,21998871122,21,998871122,True
2,VND-1003,02/02/2026,890.9,Carlos Eduardo,Fernanda Lima,fernanda.lima@invalid_email.c,31988776655,2026-02-02,31988776655,31,988776655,False
3,VND-1004,2026/02/14,5200.0,Lucas Mendes,João Pedro Santos,joao.pedro@tech.io,(41) 91234.5678,2026-02-14,41912345678,41,912345678,True
4,VND-1005,05-03-2026,1750.0,Ana Beatriz,Camila Rodrigues,camila.rodrigues@hotmail.com,(11) 97654-3210,2026-03-05,11976543210,11,976543210,True
5,VND-1006,12/03/2026,2100.3,Carlos Eduardo,Felipe Oliveira,felipe.oliveira@dominio.org,+55 (81) 98111-2233,2026-03-12,81981112233,81,981112233,True
6,VND-1007,2026-03-22,4300.0,Lucas Mendes,Beatriz Castro,beatriz.castro@br-corporation.com,85994433221,2026-03-22,85994433221,85,994433221,True
7,VND-1008,01/04/2026,650.0,Ana Beatriz,Gabriel Martins,gabriel.m@provedor.com.br,(61) 99123-8899,2026-04-01,61991238899,61,991238899,True
8,VND-1009,2026-04-10,8100.0,Carlos Eduardo,Juliana Paes,juliana.paes@invalido.x,19 98712 3456,2026-04-10,19987123456,19,987123456,False
9,VND-1010,28/04/2026,2950.8,Lucas Mendes,Thiago Barbosa,thiago.barbosa@consultoria.com,(51) 98456-7890,2026-04-28,51984567890,51,984567890,True
